In [1]:
import torch

torch.set_default_dtype(torch.float64)

from torch_openreml import MixedModelREML
from torch_openreml.covariance import (
    DiagonalMatrix,
    DummyMatrix,
    Gram,
    IdentityMatrix,
    KroneckerProduct,
    LowerTriangularMatrix,
    ScalarMatrix,
    Sum,
)
from torch_openreml.example_data import fa_met, fa_met_params

fa_met.head()

,genotype,environment,rep,yield,genetic
0,001,01,1,4.779848,-0.231039
1,001,01,2,4.818141,-0.231039
2,001,02,1,4.614127,-0.638617
3,001,02,2,4.455589,-0.638617
4,001,03,1,5.057905,0.261934


In [2]:
means = torch.tensor(fa_met_params["means"])
loadings = torch.tensor(fa_met_params["loadings"])
uniqueness = torch.tensor(fa_met_params["uniqueness"])
sigma2 = fa_met_params["sigma2"]

sigma_g = loadings @ loadings.T + torch.diag(uniqueness)
means, sigma2, sigma_g.round(decimals=4)

(tensor([5.0000, 5.2000, 4.8000, 5.1000, 5.3000, 4.9000]),
 0.3,
 tensor([[1.3100, 0.6300, 0.5400, 0.4500, 0.2700, 0.0900],
         [0.6300, 0.9300, 0.4400, 0.4300, 0.3500, 0.2300],
         [0.5400, 0.4400, 0.8700, 0.3400, 0.2500, 0.1400],
         [0.4500, 0.4300, 0.3400, 1.0100, 0.4300, 0.3700],
         [0.2700, 0.3500, 0.2500, 0.4300, 0.9800, 0.5900],
         [0.0900, 0.2300, 0.1400, 0.3700, 0.5900, 1.1500]]))

In [3]:
cells = fa_met.drop_duplicates(["genotype", "environment"])
genetic = torch.tensor(cells["genetic"].to_numpy())

genotype_labels = fa_met["genotype"]
environment_labels = fa_met["environment"]

y = torch.tensor(fa_met["yield"].to_numpy())
X = DummyMatrix(environment_labels)()

y.shape, X.shape, len(genotype_labels)

(torch.Size([1200]), torch.Size([1200, 6]), 1200)

In [4]:
Z = DummyMatrix(genotype_labels, environment_labels)

k = 2

fa = Sum(
    factors=Gram(
        LowerTriangularMatrix(environment_labels, k),
        gram_type="xxt",
    ),
    uniqueness=DiagonalMatrix(environment_labels),
)

G = KroneckerProduct(
    genotype=IdentityMatrix(genotype_labels),
    environment=fa,
)

reml = MixedModelREML(genotype=(Z, G), R=ScalarMatrix(len(y)))

In [5]:
names = reml.V.free_param_names
len(names), names

(18,
 ['G/environment/factors/op_0/L_0_0',
  'G/environment/factors/op_0/L_1_0',
  'G/environment/factors/op_0/L_1_1',
  'G/environment/factors/op_0/L_2_0',
  'G/environment/factors/op_0/L_2_1',
  'G/environment/factors/op_0/L_3_0',
  'G/environment/factors/op_0/L_3_1',
  'G/environment/factors/op_0/L_4_0',
  'G/environment/factors/op_0/L_4_1',
  'G/environment/factors/op_0/L_5_0',
  'G/environment/factors/op_0/L_5_1',
  'G/environment/uniqueness/sigma^2_0',
  'G/environment/uniqueness/sigma^2_1',
  'G/environment/uniqueness/sigma^2_2',
  'G/environment/uniqueness/sigma^2_3',
  'G/environment/uniqueness/sigma^2_4',
  'G/environment/uniqueness/sigma^2_5',
  'R/sigma^2'])

In [6]:
names, theta_hat = reml.optimize(y, X, verbose=1)

[  1] ∥∇∥:     507.6734, ∥Δ∥: 6.5757, η: 0.50, ∥Δᶜ∥: 3.2878, accepted: True , log 𝓛: -526.9347
[  2] ∥∇∥:     106.2137, ∥Δ∥: 4.1414, η: 0.50, ∥Δᶜ∥: 2.0707, accepted: True , log 𝓛: -484.2801


[  3] ∥∇∥:      89.9396, ∥Δ∥: 8.0011, η: 0.12, ∥Δᶜ∥: 1.0001, accepted: True , log 𝓛: -478.8517
[  4] ∥∇∥:      80.3064, ∥Δ∥: 6.3980, η: 0.25, ∥Δᶜ∥: 1.5995, accepted: True , log 𝓛: -472.0562


[  5] ∥∇∥:      65.9411, ∥Δ∥: 3.3889, η: 0.50, ∥Δᶜ∥: 1.6945, accepted: True , log 𝓛: -460.0205
[  6] ∥∇∥:      58.0163, ∥Δ∥: 1.4945, η: 1.00, ∥Δᶜ∥: 1.4945, accepted: True , log 𝓛: -449.7713
[  7] ∥∇∥:      59.2477, ∥Δ∥: 1.0082, η: 1.00, ∥Δᶜ∥: 1.0082, accepted: True , log 𝓛: -442.0145
[  8] ∥∇∥:      39.1265, ∥Δ∥: 0.4969, η: 1.00, ∥Δᶜ∥: 0.4969, accepted: True , log 𝓛: -434.7584


[  9] ∥∇∥:      14.5894, ∥Δ∥: 0.2143, η: 1.00, ∥Δᶜ∥: 0.2143, accepted: True , log 𝓛: -433.7947
[ 10] ∥∇∥:       4.0579, ∥Δ∥: 0.0649, η: 1.00, ∥Δᶜ∥: 0.0649, accepted: True , log 𝓛: -433.7195
[ 11] ∥∇∥:       1.2253, ∥Δ∥: 0.0314, η: 1.00, ∥Δᶜ∥: 0.0314, accepted: True , log 𝓛: -433.7115


[ 12] ∥∇∥:       0.4508, ∥Δ∥: 0.0098, η: 1.00, ∥Δᶜ∥: 0.0098, accepted: True , log 𝓛: -433.7106
[ 13] ∥∇∥:       0.1530, ∥Δ∥: 0.0038, η: 1.00, ∥Δᶜ∥: 0.0038, accepted: True , log 𝓛: -433.7105
[ 14] ∥∇∥:       0.0562, ∥Δ∥: 0.0013, η: 1.00, ∥Δᶜ∥: 0.0013, accepted: True , log 𝓛: -433.7105
[ 15] ∥∇∥:       0.0196, ∥Δ∥: 0.0005, η: 1.00, ∥Δᶜ∥: 0.0005, accepted: True , log 𝓛: -433.7105


[ 16] ∥∇∥:       0.0071, ∥Δ∥: 0.0002, η: 1.00, ∥Δᶜ∥: 0.0002, accepted: True , log 𝓛: -433.7105
[ 17] ∥∇∥:       0.0025, ∥Δ∥: 0.0001, η: 1.00, ∥Δᶜ∥: 0.0001, accepted: True , log 𝓛: -433.7105

[∇: score, Δ: 𝐉⁻¹∇, η: step length, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 17, total time: 1.42s



In [7]:
reml.is_converged(), reml.loglik()

(True, (tensor(-433.7105), 'cform'))

In [8]:
theta_hat.round(decimals=4)

tensor([ 1.0415,  0.7377, -0.4160,  0.4099, -0.4198,  0.4374, -0.4983,  0.2225,
        -0.8084,  0.1063, -0.7854, -0.6087, -0.7816, -0.4924, -0.2082, -0.3351,
        -0.2155, -0.6066])

In [9]:
params = reml.build_params(out_format="dict")
params

{'G/environment/factors/op_0/L_0_0': tensor([1.0415]),
 'G/environment/factors/op_0/L_1_0': tensor([0.7377]),
 'G/environment/factors/op_0/L_1_1': tensor([-0.4160]),
 'G/environment/factors/op_0/L_2_0': tensor([0.4099]),
 'G/environment/factors/op_0/L_2_1': tensor([-0.4198]),
 'G/environment/factors/op_0/L_3_0': tensor([0.4374]),
 'G/environment/factors/op_0/L_3_1': tensor([-0.4983]),
 'G/environment/factors/op_0/L_4_0': tensor([0.2225]),
 'G/environment/factors/op_0/L_4_1': tensor([-0.8084]),
 'G/environment/factors/op_0/L_5_0': tensor([0.1063]),
 'G/environment/factors/op_0/L_5_1': tensor([-0.7854]),
 'G/environment/uniqueness/sigma^2_0': tensor([0.2960]),
 'G/environment/uniqueness/sigma^2_1': tensor([0.2095]),
 'G/environment/uniqueness/sigma^2_2': tensor([0.3735]),
 'G/environment/uniqueness/sigma^2_3': tensor([0.6594]),
 'G/environment/uniqueness/sigma^2_4': tensor([0.5116]),
 'G/environment/uniqueness/sigma^2_5': tensor([0.6499]),
 'R/sigma^2': tensor([0.2972])}

In [10]:
tree = reml.tree()

Sigma_hat = tree["G/environment"]
Lambda_hat = tree["G/environment/factors/op_0"]
Psi_hat = tree["G/environment/uniqueness"]

Sigma_hat.round(decimals=4)

tensor([[1.3808, 0.7683, 0.4269, 0.4556, 0.2317, 0.1107],
        [0.7683, 0.9267, 0.4770, 0.5300, 0.5004, 0.4052],
        [0.4269, 0.4770, 0.7177, 0.3885, 0.4305, 0.3732],
        [0.4556, 0.5300, 0.3885, 1.0991, 0.5002, 0.4379],
        [0.2317, 0.5004, 0.4305, 0.5002, 1.2146, 0.6586],
        [0.1107, 0.4052, 0.3732, 0.4379, 0.6586, 1.2780]])

In [11]:
Lambda_hat.round(decimals=4)

tensor([[ 1.0415,  0.0000],
        [ 0.7377, -0.4160],
        [ 0.4099, -0.4198],
        [ 0.4374, -0.4983],
        [ 0.2225, -0.8084],
        [ 0.1063, -0.7854]])

In [12]:
psi_hat = Psi_hat.diagonal()
uniqueness.round(decimals=4), psi_hat.round(decimals=4)

(tensor([0.5000, 0.4000, 0.5000, 0.6000, 0.4000, 0.5000]),
 tensor([0.2960, 0.2095, 0.3735, 0.6594, 0.5116, 0.6499]))

In [13]:
sigma2_hat = params["R/sigma^2"]
sigma2_hat

tensor([0.2972])

In [14]:
sign = torch.sign(torch.diagonal(Lambda_hat))
Lambda_aligned = Lambda_hat * sign

Lambda_aligned.round(decimals=4)

tensor([[1.0415, -0.0000],
        [0.7377, 0.4160],
        [0.4099, 0.4198],
        [0.4374, 0.4983],
        [0.2225, 0.8084],
        [0.1063, 0.7854]])

In [15]:
loadings

tensor([[0.9000, 0.0000],
        [0.7000, 0.2000],
        [0.6000, 0.1000],
        [0.5000, 0.4000],
        [0.3000, 0.7000],
        [0.1000, 0.8000]])

In [16]:
(Sigma_hat - sigma_g).abs().max()

tensor(0.2346)

In [17]:
torch.corrcoef(torch.stack([Sigma_hat.flatten(), sigma_g.flatten()]))[0, 1]

tensor(0.9490)

In [18]:
blup = reml.blup_per_term()["genotype"]

torch.corrcoef(torch.stack([genetic, blup]))[0, 1]

tensor(0.9361)

In [19]:
(blup - genetic).pow(2).mean().sqrt()

tensor(0.3777)

In [20]:
beta_hat = reml.blue()
se_beta = torch.sqrt(reml.var_blue())

beta_hat.round(decimals=4), se_beta.round(decimals=4)

(tensor([5.2067, 5.4320, 4.7818, 5.1043, 5.3627, 4.9291]),
 tensor([0.1237, 0.1037, 0.0931, 0.1117, 0.1168, 0.1194]))

In [21]:
def fit_fa(k):
    fa = Sum(
        factors=Gram(
            LowerTriangularMatrix(environment_labels, k),
            gram_type="xxt",
        ),
        uniqueness=DiagonalMatrix(environment_labels),
    )
    G = KroneckerProduct(
        genotype=IdentityMatrix(genotype_labels),
        environment=fa,
    )
    model = MixedModelREML(genotype=(Z, G), R=ScalarMatrix(len(y)))
    names_k, _ = model.optimize(y, X)
    loglik_k = model.loglik()[0].item()
    p_k = len(names_k)
    return p_k, loglik_k, -2 * loglik_k + 2 * p_k

for k_k in (1, 2):
    p_k, loglik_k, aic_k = fit_fa(k_k)
    print(f"k = {k_k}: p = {p_k}, loglik = {loglik_k:.3f}, AIC = {aic_k:.3f}")

k = 1: p = 13, loglik = -448.071, AIC = 922.143


k = 2: p = 18, loglik = -433.710, AIC = 903.421


In [22]:
env_levels = sorted(fa_met["environment"].unique())
geno_levels = sorted(fa_met["genotype"].unique())

generator = torch.Generator().manual_seed(20261120)
missing = {}
for genotype in geno_levels:
    picks = torch.randperm(len(env_levels), generator=generator)[:3]
    missing[genotype] = {env_levels[i] for i in picks.tolist()}

keep = [
    environment not in missing[genotype]
    for genotype, environment in zip(fa_met["genotype"], fa_met["environment"])
]
unbalanced = fa_met[keep].reset_index(drop=True)

unbalanced.head()

,genotype,environment,rep,yield,genetic
0,001,02,1,4.614127,-0.638617
1,001,02,2,4.455589,-0.638617
2,001,04,1,5.997604,0.150623
3,001,04,2,4.610560,0.150623
4,001,05,1,4.631243,-0.792489


In [23]:
unbalanced_cells = unbalanced.drop_duplicates(["genotype", "environment"])
len(unbalanced), len(unbalanced_cells), unbalanced_cells.groupby("environment").size()

(600,
 300,
 environment
 01    51
 02    55
 03    42
 04    51
 05    48
 06    53
 dtype: int64)

In [24]:
y_u = torch.tensor(unbalanced["yield"].to_numpy())
X_u = DummyMatrix(unbalanced["environment"], levels=[env_levels])()
Z_u = DummyMatrix(
    unbalanced["genotype"],
    unbalanced["environment"],
    levels=[geno_levels, env_levels],
)

Z_u().shape, (Z_u() == 0).all(dim=0).sum()

(torch.Size([600, 600]), tensor(300))

In [25]:
fa_u = Sum(
    factors=Gram(LowerTriangularMatrix(env_levels, 2), gram_type="xxt"),
    uniqueness=DiagonalMatrix(env_levels),
)
G_u = KroneckerProduct(
    genotype=IdentityMatrix(geno_levels),
    environment=fa_u,
)

model_u = MixedModelREML(genotype=(Z_u, G_u), R=ScalarMatrix(len(y_u)))
len(model_u.V.free_param_names)

18

In [26]:
names_u, theta_u = model_u.optimize(y_u, X_u)

model_u.is_converged(), model_u.loglik()

(True, (tensor(-267.2773), 'cform'))

In [27]:
sigma2_u = model_u.build_params(out_format="dict")["R/sigma^2"]
sigma2_u

tensor([0.3389])

In [28]:
tree_u = model_u.tree()
Sigma_u = tree_u["G/environment"]
psi_u = tree_u["G/environment/uniqueness"].diagonal()

In [29]:
torch.corrcoef(torch.stack([Sigma_u.flatten(), sigma_g.flatten()]))[0, 1]

tensor(0.9045)

In [30]:
(Sigma_u - sigma_g).abs().max()

tensor(0.3837)

In [31]:
psi_u.round(decimals=4), uniqueness.round(decimals=4)

(tensor([0.2305, 0.4881, 0.1667, 0.6493, 0.0000, 0.8802]),
 tensor([0.5000, 0.4000, 0.5000, 0.6000, 0.4000, 0.5000]))

In [32]:
blup_u = model_u.blup_per_term()["genotype"]
seen = (Z_u() != 0).any(dim=0)
seen.sum()

tensor(300)

In [33]:
def corr(x, y):
    return torch.corrcoef(torch.stack([x, y]))[0, 1]

def rmse(x, y):
    return (x - y).pow(2).mean().sqrt()

In [34]:
corr(genetic[seen], blup_u[seen])

tensor(0.9366)

In [35]:
rmse(genetic[seen], blup_u[seen])

tensor(0.3722)

In [36]:
corr(genetic[~seen], blup_u[~seen])

tensor(0.5017)

In [37]:
rmse(genetic[~seen], blup_u[~seen])

tensor(0.8858)

In [38]:
blup_u[~seen].std(), genetic[~seen].std()

(tensor(0.5971), tensor(1.0037))

In [39]:
def fit_fa_u(k):
    fa = Sum(
        factors=Gram(
            LowerTriangularMatrix(env_levels, k),
            gram_type="xxt",
        ),
        uniqueness=DiagonalMatrix(env_levels),
    )
    G = KroneckerProduct(
        genotype=IdentityMatrix(geno_levels),
        environment=fa,
    )
    model = MixedModelREML(genotype=(Z_u, G), R=ScalarMatrix(len(y_u)))
    names_k, _ = model.optimize(y_u, X_u)
    loglik_k = model.loglik()[0].item()
    p_k = len(names_k)
    return p_k, loglik_k, -2 * loglik_k + 2 * p_k

p_1, loglik_1, aic_1 = fit_fa_u(1)
print(f"k = 1: p = {p_1}, loglik = {loglik_1:.3f}, AIC = {aic_1:.3f}")
print(f"k = 2: p = {len(names_u)}, loglik = {model_u.loglik()[0].item():.3f}, "
      f"AIC = {-2 * model_u.loglik()[0].item() + 2 * len(names_u):.3f}")

k = 1: p = 13, loglik = -270.979, AIC = 567.959
k = 2: p = 18, loglik = -267.277, AIC = 570.555
